In [1]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

In [5]:
url = 'https://web.archive.org/web/2021/https://www.ccny.cuny.edu/registrar/fall'
r = requests.get(url)
print(r.status_code)

soup = BeautifulSoup(r.text, 'html.parser')
table = soup.find('table')
rows = table.find_all('tr')
len(rows)

200


37

In [6]:
def parse_date(s):
    # "August 25 - 31" -> use the start date "August 25"
    # "January 1, 2022" -> has its own year
    m = re.match(r'([A-Za-z]+)\s+(\d+)(?:,\s*(\d{4}))?', s)
    if m is None:
        return None  # header row
    month, day, year = m.groups()
    return datetime.strptime(f"{month} {day} {year or 2021}", "%B %d %Y").date()

records = []
for row in rows:
    cells = row.find_all(['td', 'th'])
    if len(cells) < 3:
        continue
    date = parse_date(cells[0].get_text(strip=True))
    if date is None:
        continue
    records.append({
        'date': date,
        'dow': cells[1].get_text(strip=True),
        'text': ' '.join(cells[2].get_text().split()).rstrip(';'),
    })

len(records)

36